# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Each record set, field, and column has a unique `@id`, which is needed for loading and referencing the right data with `mlcroissant`.

In [ ]:
# List all available record sets and their fields (by @id). The .record_sets property returns Croissant RecordSet objects.
record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for idx, rs in enumerate(record_sets):
    print(f"[{idx}] Record Set @id: {rs.id}")
    if hasattr(rs, 'fields'):
        for fid, field in enumerate(rs.fields):
            print(f"    [Field {fid}] Name: {field.name} | @id: {field.id} | Data type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. For demonstration, we'll load the *first* record set found.

In [ ]:
# Extract data from each record set
dataframes = {}

# List record set @ids
record_set_ids = [rs.id for rs in record_sets]
print(f"Record Sets @ids: {record_set_ids}")

# We'll extract the first (main) record set, for illustration
main_record_set_id = record_set_ids[0]
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
dataframes[main_record_set_id] = df
print(f"Fields (columns) in main record set '{main_record_set_id}':\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

We'll pick a numeric field (e.g., MW [molecular weight] or Protein coverage %) and a categorical/grouping field. Refer to the column names printed above.

In [ ]:
main_df = dataframes[main_record_set_id]
# Attempt to find a representative numeric column for filtering - e.g., 'MW [kDa]' or similar
candidate_numeric_cols = [col for col in main_df.columns if ('MW' in col or 'Coverage' in col or main_df[col].dtype in ['float64','int64'])]
if candidate_numeric_cols:
    numeric_field = candidate_numeric_cols[0]
else:
    # fallback: any numeric-like column
    numeric_field = main_df.select_dtypes(include=['number']).columns[0]

print(f"Using numeric field: {numeric_field}")

threshold = main_df[numeric_field].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field]) else 10

# Filtering (demonstration)
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalizing the numeric field for filtered records
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt to group by a categorical field, pick the first one that has low unique count
possible_cats = [col for col in main_df.columns if main_df[col].dtype == 'object' and main_df[col].nunique() < 10]
if possible_cats:
    group_field = possible_cats[0]
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable categorical group field with <10 unique values found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll plot the distribution of the selected numeric field, and if available, a grouped boxplot by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Histogram of the main numeric field
plt.figure(figsize=(6,4))
sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If a grouping field is available, visualize boxplot
if 'group_field' in locals():
    if group_field in main_df.columns and main_df[group_field].nunique() < 20:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=main_df, showfliers=False)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id`. You can adapt the notebook to further analyze specific fields, filter based on other criteria, or join multiple record sets as needed for your scientific exploration.